# Base model test for the query generator (planner)

Tests the model **before fine-tuning** on the same `test.jsonl`, with the same system prompt, the same generation settings and the same metrics as the fine-tuned model in `model.ipynb` (cells 45 to 50), so the two sets of numbers can be compared directly.

**Before you run it**
1. Stop the Docker containers (and any other notebook) that hold the GPU. The 9B model in 4 bit needs about 7 GB; the check in the third cell prints how much is free.
2. Restart the kernel, then run the cells from top to bottom. Nothing has to be installed if `model.ipynb` already ran in this environment (Unsloth, transformers, sentence-transformers).
3. Keep `test.jsonl` in the same folder as this notebook (it is written by cell 26 of `model.ipynb`). `train.jsonl` is needed only for the few-shot option.

**How to read the result.** A base model has not been taught the output format, so the first thing to look at is the share of answers that are valid JSON with both keys. The similarity numbers are shown twice: over valid answers only (as `model.ipynb` does) and over all rows, where an invalid answer counts as 0. Only the second one is fair when the valid rates differ a lot.

The run is **resumable**: every answer is written to a checkpoint file at once, and a rerun continues where it stopped.

In [ ]:
# ---- settings: change these, then run the notebook from top to bottom ----
BASE_MODEL_ID    = "Qwen/Qwen3.5-9B-Base"  # the model that was fine-tuned (model.ipynb, cell 10). For the post-trained model use "Qwen/Qwen3.5-9B"
LOAD_IN_4BIT     = True                     # the fine-tuned model was also tested in 4 bit (model.ipynb, cell 45)
MAX_SEQ_LENGTH   = 4096                     # the fine-tuned test used 2048; the room is needed only for the few-shot prompts
NUM_SHOTS        = 0                        # 0 = zero-shot (same prompt as the fine-tuned test); 2 = two worked examples from train.jsonl in the prompt
NUM_SAMPLES      = 300                      # first N rows of test.jsonl (it was shuffled with seed 42, so this is a random sample). None = all rows
MAX_NEW_TOKENS   = 768                      # as generate_prediction in model.ipynb, cell 47
TEMPERATURE      = 0.3
TOP_P            = 0.9
DISABLE_THINKING = True                     # appends an empty <think></think> block to the prompt, as model.ipynb does
SEED             = 3407
TEST_FILE        = "test.jsonl"
TRAIN_FILE       = "train.jsonl"            # only used when NUM_SHOTS > 0
FINE_TUNED_RESULTS = "eval_results.json"    # saved by model.ipynb, cell 50; used for the comparison at the end (optional)

RUN_TAG = f"{BASE_MODEL_ID.split('/')[-1]}_{NUM_SHOTS}shot"
CHECKPOINT_PATH = f"eval_checkpoint_{RUN_TAG}.jsonl"
RESULTS_PATH    = f"eval_results_{RUN_TAG}.json"
print("run tag:", RUN_TAG)

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"        # as in model.ipynb, cell 8

from unsloth import FastModel                  # unsloth first, before transformers
import json, re, time, random, difflib
import numpy as np
import torch
from tqdm.auto import tqdm

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU found. Restart the kernel and check that the Docker containers are stopped."
free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)} | free {free / 1e9:.1f} GB of {total / 1e9:.1f} GB")
if free / 1e9 < 9:
    print("WARNING: less than 9 GB free. Stop the Docker containers or other notebooks that use the GPU, restart the kernel and run again.")

In [ ]:
# Load the base model (no LoRA adapter, no fine-tuning) the same way model.ipynb loads the fine-tuned one.
model, tokenizers = FastModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
FastModel.for_inference(model)
print("loaded", BASE_MODEL_ID, "| LoRA adapter present:", any("lora" in n.lower() for n, _ in model.named_parameters()))

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are a research question decomposition agent. Given a research "
    "question, break it into topic-grouped sub-questions with detail "
    "questions, and provide search ready queries."
)

assert os.path.exists(TEST_FILE), f"{TEST_FILE} not found next to this notebook. It is written by cell 26 of model.ipynb."
test_dataset = load_dataset("json", data_files=TEST_FILE)["train"]
n_all = len(test_dataset)
if NUM_SAMPLES is not None:
    test_dataset = test_dataset.select(range(min(NUM_SAMPLES, n_all)))
print(f"test rows: {len(test_dataset)} of {n_all}")

# the file must be the one the fine-tuned model was tested on: chat format, the same system prompt
row = test_dataset[0]["messages"]
assert [m["role"] for m in row] == ["system", "user", "assistant"], "unexpected format: expected system, user, assistant messages"
assert row[0]["content"] == SYSTEM_PROMPT, "the system prompt in test.jsonl differs from the one used for the fine-tuned test"
print("question :", row[1]["content"])
print("reference:", row[2]["content"][:300], "...")

In [ ]:
# Few-shot examples (NUM_SHOTS > 0): worked examples from train.jsonl, short ones, chosen with a fixed seed.
def pick_shots(n, max_chars=1800):
    if n == 0:
        return []
    assert os.path.exists(TRAIN_FILE), f"{TRAIN_FILE} is needed for NUM_SHOTS > 0"
    train = load_dataset("json", data_files=TRAIN_FILE)["train"]
    order = list(range(len(train)))
    random.Random(SEED).shuffle(order)
    shots = []
    for i in order:
        msgs = train[i]["messages"]
        if len(msgs[2]["content"]) <= max_chars:
            shots.append((msgs[1]["content"], msgs[2]["content"]))
            if len(shots) == n:
                break
    return shots

SHOTS = pick_shots(NUM_SHOTS)


def build_messages(question):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for q, a in SHOTS:
        messages += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    messages.append({"role": "user", "content": question})
    return messages


def generate_prediction(question, max_new_tokens=MAX_NEW_TOKENS):
    # Same as generate_prediction in model.ipynb (cell 47), plus the time and the number of new tokens.
    text = tokenizers.apply_chat_template(build_messages(question), tokenize=False, add_generation_prompt=True)
    if DISABLE_THINKING:
        text += "<think>\n\n</think>\n\n"
    inputs = tokenizers(text=text, return_tensors="pt").to("cuda")
    n_in = inputs["input_ids"].shape[1]
    t0 = time.time()
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        do_sample=True,
    )
    seconds = time.time() - t0
    new_tokens = int(outputs.shape[1] - n_in)
    raw = tokenizers.decode(outputs[0][n_in:], skip_special_tokens=True).strip()
    return raw, seconds, new_tokens


print(f"shots: {len(SHOTS)}")

In [ ]:
# Smoke test on 3 questions: look at what the base model actually writes before running the whole test.
torch.manual_seed(SEED)
for i in range(3):
    q = test_dataset[i]["messages"][1]["content"]
    raw, sec, ntok = generate_prediction(q)
    print(f"[{i}] {q}\n    {ntok} new tokens in {sec:.1f} s\n    {raw[:600]!r}\n")

In [ ]:
# ---- metrics: the same code as model.ipynb, cell 49 (the cosine uses numpy on normalised vectors, which gives the same value as scikit-learn) ----
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")


def extract_json_from_output(raw_output: str):
    raw_output = raw_output.strip()
    decoder = json.JSONDecoder()
    start = raw_output.find("{")
    if start == -1:
        return None
    try:
        obj, _ = decoder.raw_decode(raw_output[start:])
        return obj
    except json.JSONDecodeError:
        return None


def flatten_topics_only(parsed_json):
    if not parsed_json or "sub_questions" not in parsed_json:
        return []
    topics = []
    for item in parsed_json["sub_questions"]:
        if isinstance(item, dict) and "topic" in item:
            topics.append(item["topic"])
        elif isinstance(item, str):
            topics.append(item)
    return topics


def flatten_details_only(parsed_json):
    if not parsed_json or "sub_questions" not in parsed_json:
        return []
    all_details = []
    for item in parsed_json["sub_questions"]:
        if isinstance(item, dict):
            details = item.get("details", [])
            if isinstance(details, list):
                all_details.extend(str(d) for d in details)
    return all_details


def flatten_queries(parsed_json):
    if not parsed_json or "search_queries" not in parsed_json:
        return []
    return [q for q in parsed_json["search_queries"] if isinstance(q, str)]


def embedding_cosine_similarity(text_a: str, text_b: str) -> float:
    if not text_a.strip() or not text_b.strip():
        return 0.0
    e = embedder.encode([text_a, text_b], normalize_embeddings=True)
    return float(np.dot(e[0], e[1]))


def text_overlap_ratio(text_a: str, text_b: str) -> float:
    return difflib.SequenceMatcher(None, text_a.lower(), text_b.lower()).ratio()


def evaluate_single(question: str, generated_raw: str, reference_raw: str):
    gen_parsed = extract_json_from_output(generated_raw)
    ref_parsed = extract_json_from_output(reference_raw)

    result = {
        "question": question,
        "valid_json": gen_parsed is not None,
    }

    if gen_parsed is None or ref_parsed is None:
        result.update({
            "topic_similarity": 0.0,
            "details_similarity": 0.0,
            "query_similarity": 0.0,
            "topic_count_match": False,
            "query_count_match": False,
        })
        return result

    gen_topics = flatten_topics_only(gen_parsed)
    ref_topics = flatten_topics_only(ref_parsed)
    gen_details = flatten_details_only(gen_parsed)
    ref_details = flatten_details_only(ref_parsed)
    gen_queries = flatten_queries(gen_parsed)
    ref_queries = flatten_queries(ref_parsed)

    gen_topics_text = " ".join(gen_topics)
    ref_topics_text = " ".join(ref_topics)
    gen_details_text = " ".join(gen_details)
    ref_details_text = " ".join(ref_details)
    gen_queries_text = " ".join(gen_queries)
    ref_queries_text = " ".join(ref_queries)

    result.update({
        "num_sub_questions_generated": len(gen_topics),
        "num_sub_questions_reference": len(ref_topics),
        "num_details_generated": len(gen_details),
        "num_details_reference": len(ref_details),
        "num_search_queries_generated": len(gen_queries),
        "num_search_queries_reference": len(ref_queries),
        "topic_count_match": len(gen_topics) == len(ref_topics),
        "query_count_match": len(gen_queries) == len(ref_queries),
        "topic_similarity": embedding_cosine_similarity(gen_topics_text, ref_topics_text),
        "details_similarity": embedding_cosine_similarity(gen_details_text, ref_details_text),
        "query_similarity": embedding_cosine_similarity(gen_queries_text, ref_queries_text),
        "topic_text_overlap": text_overlap_ratio(gen_topics_text, ref_topics_text),
        "details_text_overlap": text_overlap_ratio(gen_details_text, ref_details_text),
        "query_text_overlap": text_overlap_ratio(gen_queries_text, ref_queries_text),
        "generated_topics": gen_topics,
        "reference_topics": ref_topics,
        "generated_details": gen_details,
        "reference_details": ref_details,
        "generated_queries": gen_queries,
        "reference_queries": ref_queries,
    })
    return result


def schema_ok(raw_output: str) -> bool:
    # Extra check for a base model: valid JSON is not enough, it must have both keys with non-empty lists inside.
    p = extract_json_from_output(raw_output)
    return (isinstance(p, dict) and isinstance(p.get("sub_questions"), list) and len(p["sub_questions"]) > 0
            and isinstance(p.get("search_queries"), list) and len(p["search_queries"]) > 0)

In [ ]:
def summarize(results, title="", show=True):
    # Numbers as in model.ipynb (averages over the valid answers), plus the same averages over ALL rows (an invalid answer counts as 0)
    # and the share of answers that have the full structure.
    n_total = len(results)
    valid = [r for r in results if r["valid_json"]]
    n_valid = len(valid)
    keys = ["topic_similarity", "details_similarity", "query_similarity"]
    out = {
        "rows": n_total,
        "valid_json_rate": n_valid / n_total if n_total else 0.0,
        "schema_ok_rate": sum(1 for r in results if r.get("schema_ok", r["valid_json"])) / n_total if n_total else 0.0,
    }
    for k in keys:
        out[k + "_valid_only"] = sum(r[k] for r in valid) / n_valid if n_valid else 0.0
        out[k + "_all_rows"] = sum(r[k] for r in results) / n_total if n_total else 0.0
    out["topic_count_match_rate_valid_only"] = sum(r["topic_count_match"] for r in valid) / n_valid if n_valid else 0.0
    out["query_count_match_rate_valid_only"] = sum(r["query_count_match"] for r in valid) / n_valid if n_valid else 0.0
    out["topic_count_match_rate_all_rows"] = sum(r["topic_count_match"] for r in results) / n_total if n_total else 0.0
    out["query_count_match_rate_all_rows"] = sum(r["query_count_match"] for r in results) / n_total if n_total else 0.0
    secs = [r["seconds"] for r in results if "seconds" in r]
    toks = [r["new_tokens"] for r in results if "new_tokens" in r]
    if secs:
        out["avg_seconds_per_answer"] = sum(secs) / len(secs)
        out["avg_new_tokens"] = sum(toks) / len(toks)
        out["hit_token_limit_rate"] = sum(1 for t in toks if t >= MAX_NEW_TOKENS) / len(toks)
    if show:
        print("=" * 64)
        print(title or "Summary")
        print(f"Rows evaluated:                    {n_total}")
        print(f"Valid JSON rate:                   {n_valid}/{n_total} ({100 * out['valid_json_rate']:.1f}%)")
        print(f"Valid JSON with both keys:         {100 * out['schema_ok_rate']:.1f}%")
        print(f"Avg topic similarity:              {out['topic_similarity_valid_only']:.3f}  (valid only)   {out['topic_similarity_all_rows']:.3f}  (all rows)")
        print(f"Avg details similarity:            {out['details_similarity_valid_only']:.3f}  (valid only)   {out['details_similarity_all_rows']:.3f}  (all rows)")
        print(f"Avg query similarity:              {out['query_similarity_valid_only']:.3f}  (valid only)   {out['query_similarity_all_rows']:.3f}  (all rows)")
        print(f"Topic count match rate:            {100 * out['topic_count_match_rate_valid_only']:.1f}%  (valid only)   {100 * out['topic_count_match_rate_all_rows']:.1f}%  (all rows)")
        print(f"Query count match rate:            {100 * out['query_count_match_rate_valid_only']:.1f}%  (valid only)   {100 * out['query_count_match_rate_all_rows']:.1f}%  (all rows)")
        if secs:
            print(f"Avg time per answer:               {out['avg_seconds_per_answer']:.1f} s, {out['avg_new_tokens']:.0f} new tokens, {100 * out['hit_token_limit_rate']:.1f}% hit the {MAX_NEW_TOKENS}-token limit")
        print("=" * 64)
    return out


def run_evaluation(dataset, checkpoint_path=CHECKPOINT_PATH, resume=True, verbose=False):
    # As run_evaluation in model.ipynb (cell 49): one result per line is written to the checkpoint at once, and a rerun skips what is done.
    results, done = [], set()
    if resume and os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue                      # a half-written last line after a crash
                results.append(r)
                done.add(r["question"])
        if results:
            print(f"Resuming: {len(results)} rows already in {checkpoint_path}")
    torch.manual_seed(SEED)
    with open(checkpoint_path, "a", encoding="utf-8") as f:
        for i in tqdm(range(len(dataset)), desc="Evaluating"):
            row = dataset[i]
            question = row["messages"][1]["content"]
            reference_raw = row["messages"][2]["content"]
            if question in done:
                continue
            generated_raw, seconds, new_tokens = generate_prediction(question)
            r = evaluate_single(question, generated_raw, reference_raw)
            r.update({"schema_ok": schema_ok(generated_raw), "seconds": round(seconds, 2), "new_tokens": new_tokens,
                      "generated_raw": generated_raw, "reference_raw": reference_raw})
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())
            results.append(r)
            done.add(question)
            if verbose:
                print(f"[{i}] valid={r['valid_json']} schema_ok={r['schema_ok']} topic={r['topic_similarity']:.3f} query={r['query_similarity']:.3f}")
    return results

In [ ]:
# The test itself. A first pass of 300 rows is enough to see the picture; set NUM_SAMPLES = None in the settings cell (and run from the top) for all rows.
eval_results = run_evaluation(test_dataset)
base_summary = summarize(eval_results, title=f"BASE MODEL: {BASE_MODEL_ID}, {NUM_SHOTS}-shot")

In [ ]:
# Why do the invalid answers fail? (the part that explains the numbers)
from collections import Counter

def failure_kind(r):
    raw = r["generated_raw"]
    if r.get("schema_ok"):
        return "ok: valid JSON with both keys"
    if r["valid_json"]:
        return "valid JSON but wrong or missing keys"
    if "{" not in raw:
        return "no JSON at all (prose, a list, or empty)"
    return "JSON started but broken or cut off"

kinds = Counter(failure_kind(r) for r in eval_results)
for k, v in kinds.most_common():
    print(f"{v:5d}  {100 * v / len(eval_results):5.1f}%  {k}")

shown = Counter()
print()
for r in eval_results:
    k = failure_kind(r)
    if k.startswith("ok") or shown[k] >= 2:
        continue
    shown[k] += 1
    print(f"--- {k}\nquestion: {r['question']}\nanswer  : {r['generated_raw'][:500]!r}\n")

In [ ]:
# Comparison with the fine-tuned model (model.ipynb, cell 50: 4,821 test rows, checkpoint-32538).
FINE_TUNED_FROM_NOTEBOOK = {   # the printed summary of model.ipynb, cell 50 (valid answers only)
    "rows": 4821, "valid_json_rate": 4801 / 4821, "topic_similarity_valid_only": 0.901, "details_similarity_valid_only": 0.838,
    "query_similarity_valid_only": 0.894, "topic_count_match_rate_valid_only": 0.452, "query_count_match_rate_valid_only": 0.106,
}
ft = None
if os.path.exists(FINE_TUNED_RESULTS):
    ft_all = json.load(open(FINE_TUNED_RESULTS, encoding="utf-8"))
    questions = {r["question"] for r in eval_results}
    ft_rows = [r for r in ft_all if r["question"] in questions]          # the same questions as the base model got
    if ft_rows:
        ft = summarize(ft_rows, show=False)
        print(f"fine-tuned model on the same {len(ft_rows)} questions (from {FINE_TUNED_RESULTS})")
if ft is None:
    ft = FINE_TUNED_FROM_NOTEBOOK
    print(f"WARNING: {FINE_TUNED_RESULTS} not found or no common questions: using the printed numbers of model.ipynb over all 4,821 rows (valid answers only).")

def pct(x):
    return "n/a" if x is None else f"{100 * x:.1f}%"

def num(x):
    return "n/a" if x is None else f"{x:.3f}"

rows = [
    ("Valid JSON rate",                            pct(base_summary["valid_json_rate"]),                   pct(ft.get("valid_json_rate"))),
    ("Valid JSON with both keys",                  pct(base_summary["schema_ok_rate"]),                    pct(ft.get("schema_ok_rate"))),
    ("Topic similarity (valid answers only)",      num(base_summary["topic_similarity_valid_only"]),       num(ft.get("topic_similarity_valid_only"))),
    ("Details similarity (valid answers only)",    num(base_summary["details_similarity_valid_only"]),     num(ft.get("details_similarity_valid_only"))),
    ("Query similarity (valid answers only)",      num(base_summary["query_similarity_valid_only"]),       num(ft.get("query_similarity_valid_only"))),
    ("Topic similarity (all rows, invalid = 0)",   num(base_summary["topic_similarity_all_rows"]),         num(ft.get("topic_similarity_all_rows"))),
    ("Details similarity (all rows, invalid = 0)", num(base_summary["details_similarity_all_rows"]),       num(ft.get("details_similarity_all_rows"))),
    ("Query similarity (all rows, invalid = 0)",   num(base_summary["query_similarity_all_rows"]),         num(ft.get("query_similarity_all_rows"))),
    ("Topic count match (valid answers only)",     pct(base_summary["topic_count_match_rate_valid_only"]), pct(ft.get("topic_count_match_rate_valid_only"))),
    ("Query count match (valid answers only)",     pct(base_summary["query_count_match_rate_valid_only"]), pct(ft.get("query_count_match_rate_valid_only"))),
]
print(f"\n{'':46s}{'base model':>14s}{'fine-tuned':>14s}")
for name, b, f_ in rows:
    print(f"{name:46s}{b:>14s}{f_:>14s}")

In [ ]:
# Save everything and free the GPU.
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump({"settings": {"model": BASE_MODEL_ID, "shots": NUM_SHOTS, "samples": len(eval_results), "max_new_tokens": MAX_NEW_TOKENS,
                            "temperature": TEMPERATURE, "top_p": TOP_P, "load_in_4bit": LOAD_IN_4BIT, "seed": SEED},
               "summary": base_summary, "results": eval_results}, f, indent=2, ensure_ascii=False)
print("saved", RESULTS_PATH)

import gc
for name in ["model", "tokenizers", "embedder"]:
    if name in globals():
        del globals()[name]
for _ in range(3):
    gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 1e9:.2f} GB")